# Capítulo 6 · Agrupamiento jerárquico aglomerativo

¿Dónde termina un grupo y empieza otro? En las dos lunas podemos seguir una curva con la vista. En Iris tenemos cuatro medidas por flor y resulta más difícil decidir qué flores se parecen. En **aprendizaje no supervisado**, el algoritmo propone grupos a partir de las características. Guardaremos las etiquetas conocidas para interpretar los resultados después del ajuste.

Al terminar podrás explicar cómo se fusionan los grupos, leer un corte en un dendrograma y reconocer cuándo el criterio de enlace cambia el resultado.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import load_iris, make_moons
from sklearn.preprocessing import StandardScaler

## 1. Dos lunas y tres maneras de medir la distancia entre grupos

Al principio cada punto forma su propio grupo. En cada paso se fusionan los dos grupos más cercanos. El **enlace** define cómo calcular esa cercanía. Imagina que dibujas una línea desde cada punto del primer grupo hasta cada punto del segundo.

| Enlace | ¿Qué distancia usamos entre los dos grupos? |
| --- | --- |
| Único (`single`) | La longitud de la línea más corta. |
| Completo (`complete`) | La longitud de la línea más larga. |
| Promedio (`average`) | La media de las longitudes de todas esas líneas. |

Por ejemplo, si las distancias entre los pares de puntos de dos grupos son 1, 2, 4 y 5, el enlace único da **1**, el completo da **5** y el promedio da **3**. En los tres casos se elige el par de grupos con el valor más pequeño según el enlace elegido.

Generamos 200 puntos en dos arcos. Si dos arcos tienen sus extremos cerca pero sus demás puntos están lejos, ¿cuál de los tres enlaces sería más sensible a esos extremos?

In [ ]:
X_lunas, luna_origen = make_moons(n_samples=200, noise=0.08, random_state=42)

In [ ]:
unico = AgglomerativeClustering(n_clusters=2, linkage="single")
completo = AgglomerativeClustering(n_clusters=2, linkage="complete")
promedio = AgglomerativeClustering(n_clusters=2, linkage="average")

grupos_unico = unico.fit_predict(X_lunas)
grupos_completo = completo.fit_predict(X_lunas)
grupos_promedio = promedio.fit_predict(X_lunas)

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey=True)
paneles = [
    ("Lunas de origen", luna_origen),
    ("Enlace único", grupos_unico),
    ("Enlace completo", grupos_completo),
    ("Enlace promedio", grupos_promedio),
]
for eje, (titulo, colores) in zip(ejes.flat, paneles):
    eje.scatter(X_lunas[:, 0], X_lunas[:, 1], c=colores, cmap="viridis", s=18)
    eje.set(title=titulo, aspect="equal")
fig.supxlabel("Coordenada x")
fig.supylabel("Coordenada y")
fig.tight_layout()
plt.show()

Con esta muestra, el enlace único mantiene un color a lo largo de cada arco. El completo y el promedio reúnen puntos de lunas distintas. Observa el extremo de cada arco para localizar dónde cambia de grupo. Para comparar los paneles, sigue los puntos que quedan juntos. Los números de grupo son arbitrarios, así que dos colores pueden intercambiarse entre paneles.

### ¿Dónde se ven las fusiones?

Un **dendrograma** registra toda la secuencia de uniones. Cada punto empieza en la base. La altura de una unión indica, para el enlace único, la distancia entre los vecinos que conectaron esos grupos. Dibujaremos la línea de corte entre las dos últimas alturas de fusión. Cuenta las ramas verticales que atraviesa.

In [ ]:
arbol = linkage(X_lunas, method="single")
# Cada fila representa una fusión. SciPy organiza las columnas en este orden:
# 0 y 1: identificadores de los dos grupos que se unen.
# 2: distancia de fusión (altura en el dendrograma). Con enlace único,
#    es la distancia entre los dos puntos más cercanos, uno de cada grupo.
# 3: número total de puntos que contiene el grupo recién formado.
arbol[-2:]  # Muestra las dos últimas fusiones con sus cuatro columnas.

### ¿Cómo se leen las dos últimas filas?

Con los 200 puntos de este ejemplo, la última fila se lee así (valores redondeados): `[395, 397, 0.302, 200]`. Los grupos **395** y **397** se unen a una distancia de **0.302** y forman un grupo de **200 puntos**. Los dos primeros números pueden superar 199 porque cada fusión recibe un identificador nuevo.

Imagina que recorres el dendrograma de abajo hacia arriba. A una altura de **0.230** aproximadamente ocurre la penúltima fusión y pasas de tres grupos a dos. A **0.302** ocurre la última y pasas de dos grupos a uno. **¿Cuántos grupos hay a mitad de ese recorrido?** Todavía hay dos.

Por eso usamos `arbol[-2:, 2]`: tomamos las dos últimas filas y leemos su tercera columna, la de las alturas. Su promedio, **0.266** aproximadamente, nos da una altura intermedia donde dibujar la línea horizontal.

In [ ]:
# -2: selecciona las dos últimas fusiones y 2 selecciona la columna de alturas.
# Su promedio sitúa el corte entre ambas fusiones, donde quedan dos grupos.
altura_corte = arbol[-2:, 2].mean()

fig, eje = plt.subplots(figsize=(10, 4))
dendrogram(arbol, no_labels=True, color_threshold=altura_corte, ax=eje)
eje.axhline(altura_corte, color="black", linestyle="--", label="Corte en dos grupos")
eje.set(title="Fusiones de las dos lunas con enlace único",
        xlabel="Puntos ordenados por el árbol", ylabel="Distancia de fusión")
eje.legend()
fig.tight_layout()
plt.show()

La línea cruza dos ramas, una por cada grupo del ajuste con enlace único. Si bajas el corte por debajo de la penúltima fusión, encontrarás más grupos. Una cadena de puntos ruidosos también puede unir regiones distintas, por lo que conviene mirar el árbol junto con la nube de puntos.

## 2. Iris · agrupar flores por sus medidas

Cada una de las 150 filas describe una flor mediante el largo y el ancho de su sépalo y de su pétalo, en centímetros. Aunque comparten unidad, algunas medidas varían más que otras. **Estandarizar** centra cada columna en cero y expresa sus valores en desviaciones estándar. Así damos a las cuatro medidas un peso comparable en el cálculo de distancias.

Usaremos el enlace promedio para elegir en cada paso los dos grupos con la menor distancia media entre sus flores. Pedimos tres grupos para explorar cómo se relacionan con las tres especies documentadas.

In [ ]:
iris = load_iris(as_frame=True)
medidas = iris.data.copy()
medidas.columns = ["longitud_sépalo", "ancho_sépalo", "longitud_pétalo", "ancho_pétalo"]
escalador = StandardScaler()
X_iris = escalador.fit_transform(medidas)
medidas.head()

In [ ]:
modelo_iris = AgglomerativeClustering(n_clusters=3, linkage="average")
grupos_iris = modelo_iris.fit_predict(X_iris)

fig, eje = plt.subplots(figsize=(6, 4))
for grupo in sorted(set(grupos_iris)):
    flores = medidas[grupos_iris == grupo]
    eje.scatter(flores["longitud_pétalo"], flores["ancho_pétalo"], label=f"Grupo {grupo}", alpha=0.7)
eje.set(title="Iris con enlace promedio",
        xlabel="Longitud del pétalo (cm)", ylabel="Ancho del pétalo (cm)")
eje.legend()
fig.tight_layout()
plt.show()

El modelo utilizó las cuatro medidas. La figura muestra dos en sus unidades originales, por lo que dos puntos cercanos en el dibujo pueden diferir en el sépalo. Recuperamos ahora la especie para contar cuántas flores de cada una quedaron en cada grupo.

In [ ]:
especie = iris.target_names[iris.target]
pd.crosstab(especie, grupos_iris, rownames=["Especie"], colnames=["Grupo"])

Lee la tabla por filas. Las 50 flores *setosa* quedan juntas. En otra columna aparecen las 50 *versicolor* y 47 *virginica*. Las tres *virginica* restantes forman el tercer grupo. Pedir tres grupos, por tanto, permite obtener grupos de tamaños muy distintos aunque haya 50 flores de cada especie.

**Para discutir en clase.** Si pasaras de tres a cuatro grupos con enlace promedio sobre estas mismas flores, ¿qué cambiaría en el árbol y qué cambiaría en el corte? Piensa en cuáles de las fusiones ya calculadas conservarías.